In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

print("TensorFlow version:", tf.__version__)
print("Libraries loaded successfully! ✅")

TensorFlow version: 2.21.0
Libraries loaded successfully! ✅


In [3]:
dataset_dir = "dataset"

print("Dataset path:", os.path.abspath(dataset_dir))
print("Dataset exists:", os.path.exists(dataset_dir))

Dataset path: c:\Users\jamee\OneDrive\Desktop\RECYZA\dataset
Dataset exists: True


In [5]:
image_extensions = (".jpg", ".jpeg", ".png", ".webp")

class_names = sorted([
    folder for folder in os.listdir(dataset_dir)
    if os.path.isdir(os.path.join(dataset_dir, folder))
])

print("Number of classes:", len(class_names))
print("\nClasses:")

total_images = 0

for class_name in class_names:
    class_path = os.path.join(dataset_dir, class_name)

    count = len([
        f for f in os.listdir(class_path)
        if f.lower().endswith(image_extensions)
    ])

    total_images += count

    print(f"{class_name}: {count}")

print("\nTotal images:", total_images)

Number of classes: 10

Classes:
Denver perfume: 37
Dove HairFall Rescue: 42
Livon serum: 45
Lizol floor cleaner: 31
Mamaearth Rosemary Water: 37
clinic plus shampoo: 37
nivya body mik: 36
parachute hair oil: 32
whitetone face powder: 42
wotta girl perfume: 35

Total images: 374


In [7]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\nTraining batches:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Validation batches:", tf.data.experimental.cardinality(validation_dataset).numpy())

print("\nClasses:")
print(train_dataset.class_names)

Found 129 files belonging to 10 classes.
Using 104 files for training.
Found 129 files belonging to 10 classes.
Using 25 files for validation.

Training batches: 7
Validation batches: 2

Classes:
['Denver perfume', 'Dove HairFall Rescue', 'Livon serum', 'Lizol floor cleaner', 'Mamaearth Rosemary Water', 'clinic plus shampoo', 'nivya body mik', 'parachute hair oil', 'whitetone face powder', 'wotta girl perfume']


In [8]:
from PIL import Image
import os

image_extensions = (".jpg", ".jpeg", ".png", ".webp")

converted = 0

for class_name in os.listdir(dataset_dir):

    class_path = os.path.join(dataset_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    for filename in os.listdir(class_path):

        if filename.lower().endswith(".webp"):

            old_path = os.path.join(class_path, filename)

            try:
                img = Image.open(old_path).convert("RGB")

                new_filename = os.path.splitext(filename)[0] + ".jpg"
                new_path = os.path.join(class_path, new_filename)

                img.save(new_path, "JPEG", quality=95)

                os.remove(old_path)

                converted += 1

            except Exception as e:
                print("Could not convert:", filename, e)

print("WEBP images converted to JPG:", converted)

WEBP images converted to JPG: 245


In [9]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.20,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=16,
    shuffle=True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.20,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("\nClasses:", train_dataset.class_names)
print("Number of classes:", len(train_dataset.class_names))

Found 374 files belonging to 10 classes.
Using 300 files for training.
Found 374 files belonging to 10 classes.
Using 74 files for validation.

Classes: ['Denver perfume', 'Dove HairFall Rescue', 'Livon serum', 'Lizol floor cleaner', 'Mamaearth Rosemary Water', 'clinic plus shampoo', 'nivya body mik', 'parachute hair oil', 'whitetone face powder', 'wotta girl perfume']
Number of classes: 10


In [10]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15)
])

print("Data augmentation created successfully! ✅")

Data augmentation created successfully! ✅


In [11]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.35)(x)

x = tf.keras.layers.Dense(128, activation="relu")(x)

x = tf.keras.layers.Dropout(0.25)(x)

outputs = tf.keras.layers.Dense(
    len(train_dataset.class_names),
    activation="softmax"
)(x)

recyza_model = tf.keras.Model(inputs, outputs)

print("RECYZA model created successfully! ✅")
print("Classes:", len(train_dataset.class_names))

RECYZA model created successfully! ✅
Classes: 10


In [12]:
recyza_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("RECYZA model compiled successfully! ✅")

RECYZA model compiled successfully! ✅


In [13]:
history = recyza_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            "RECYZA_BEST_10_EPOCH_MODEL.keras",
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-7,
            verbose=1
        )
    ]
)

Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 608ms/step - accuracy: 0.1045 - loss: 2.9202
Epoch 1: val_accuracy improved from None to 0.68919, saving model to RECYZA_BEST_10_EPOCH_MODEL.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 24s 855ms/step - accuracy: 0.1300 - loss: 2.6663 - val_accuracy: 0.6892 - val_loss: 1.6227 - learning_rate: 5.0000e-04
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 590ms/step - accuracy: 0.3314 - loss: 1.8982
Epoch 2: val_accuracy improved from 0.68919 to 0.72973, saving model to RECYZA_BEST_10_EPOCH_MODEL.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 15s 789ms/step - accuracy: 0.3900 - loss: 1.7944 - val_accuracy: 0.7297 - val_loss: 1.4057 - learning_rate: 5.0000e-04
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 562ms/step - accuracy: 0.5720 - loss: 1.4136
Epoch 3: val_accuracy improved from 0.72973 to 0.89189, saving model to RECYZA_BEST_10_EPOCH_MODEL.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 14s 714ms/step - accuracy: 0.5900 - loss: 1.3594 - val_accuracy: 0.8919 - val_loss: 0.7927 - learning_rate: 5.0

In [14]:
best_model = tf.keras.models.load_model(
    "RECYZA_BEST_10_EPOCH_MODEL.keras"
)

test_loss, test_accuracy = best_model.evaluate(
    test_dataset,
    verbose=1
)

print("\n========================================")
print("       RECYZA FINAL TEST RESULT")
print("========================================")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print("========================================")

NameError: name 'test_dataset' is not defined

In [15]:
test_dir = r"C:\Users\jamee\OneDrive\Desktop\RECYZA\recyza_split\test"

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("\nTest classes:")
print(test_dataset.class_names)

print("Test dataset created successfully! ✅")

Found 14 files belonging to 10 classes.

Test classes:
['Denver perfume', 'Dove HairFall Rescue', 'Livon serum', 'Lizol floor cleaner', 'Mamaearth Rosemary Water', 'clinic plus shampoo', 'nivya body mik', 'parachute hair oil', 'whitetone face powder', 'wotta girl perfume']
Test dataset created successfully! ✅


In [17]:
import os

base = r"C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset"

print("Searching for test folders...\n")

for root, dirs, files in os.walk(base):
    if os.path.basename(root).lower() == "test":
        image_count = sum(
            1 for f in files
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        )
        print(root)
        print("Images in this folder:", image_count)
        print()

Searching for test folders...



In [18]:
import os

base = r"C:\Users\jamee\OneDrive\Desktop\RECYZA"

extensions = (".jpg", ".jpeg", ".png", ".webp")

print("Searching for image folders...\n")

for root, dirs, files in os.walk(base):
    count = sum(
        1 for f in files
        if f.lower().endswith(extensions)
    )

    if count > 0:
        print(f"{root}  ->  {count} images")

Searching for image folders...

C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\clinic plus shampoo  ->  37 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\Denver perfume  ->  37 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\Dove HairFall Rescue  ->  42 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\Livon serum  ->  45 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\Lizol floor cleaner  ->  31 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\Mamaearth Rosemary Water  ->  37 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\nivya body mik  ->  36 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\parachute hair oil  ->  32 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\whitetone face powder  ->  42 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\Dataset\wotta girl perfume  ->  35 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\recyza_split\test\Denver perfume  ->  7 images
C:\Users\jamee\OneDrive\Desktop\RECYZA\recyza_split\test\Dove HairFall Res

In [19]:
from PIL import Image
import os

test_dir = r"C:\Users\jamee\OneDrive\Desktop\RECYZA\recyza_split\test"

converted = 0

for root, dirs, files in os.walk(test_dir):
    for filename in files:

        if filename.lower().endswith(".webp"):

            old_path = os.path.join(root, filename)

            try:
                img = Image.open(old_path).convert("RGB")

                new_path = os.path.splitext(old_path)[0] + ".jpg"

                img.save(new_path, "JPEG", quality=95)

                os.remove(old_path)

                converted += 1

            except Exception as e:
                print("Error:", old_path, e)

print("Test WEBP images converted:", converted)

Test WEBP images converted: 47


In [20]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("\nTest classes:")
print(test_dataset.class_names)

print("\nTest dataset ready! ✅")

Found 61 files belonging to 10 classes.

Test classes:
['Denver perfume', 'Dove HairFall Rescue', 'Livon serum', 'Lizol floor cleaner', 'Mamaearth Rosemary Water', 'clinic plus shampoo', 'nivya body mik', 'parachute hair oil', 'whitetone face powder', 'wotta girl perfume']

Test dataset ready! ✅


In [21]:
import os

extensions = (".jpg", ".jpeg", ".png")

print("Files in test folders that Keras should recognize:\n")

total = 0

for root, dirs, files in os.walk(test_dir):
    for f in files:
        if f.lower().endswith(extensions):
            total += 1
        else:
            print(os.path.join(root, f))

print("\nRecognized image files:", total)

Files in test folders that Keras should recognize:

C:\Users\jamee\OneDrive\Desktop\RECYZA\recyza_split\test\Dove HairFall Rescue\a9238b85948d4b2683f6c861c66a6e7f.avif

Recognized image files: 61


In [ ]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("\nTest classes:")
print(test_dataset.class_names)

print("\nTest dataset ready! ✅")

Found 61 files belonging to 10 classes.

Test classes:
['Denver perfume', 'Dove HairFall Rescue', 'Livon serum', 'Lizol floor cleaner', 'Mamaearth Rosemary Water', 'clinic plus shampoo', 'nivya body mik', 'parachute hair oil', 'whitetone face powder', 'wotta girl perfume']

Test dataset ready! ✅


In [22]:
from PIL import Image
import os

avif_path = r"C:\Users\jamee\OneDrive\Desktop\RECYZA\recyza_split\test\Dove HairFall Rescue\a9238b85948d4b2683f6c861c66a6e7f.avif"

img = Image.open(avif_path).convert("RGB")

jpg_path = os.path.splitext(avif_path)[0] + ".jpg"

img.save(jpg_path, "JPEG", quality=95)

os.remove(avif_path)

print("AVIF converted to JPG successfully! ✅")
print("New file:", jpg_path)

AVIF converted to JPG successfully! ✅
New file: C:\Users\jamee\OneDrive\Desktop\RECYZA\recyza_split\test\Dove HairFall Rescue\a9238b85948d4b2683f6c861c66a6e7f.jpg


In [23]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("\n========================================")
print("FINAL TEST DATASET")
print("========================================")
print("Test images:", tf.data.experimental.cardinality(test_dataset).numpy() * 16)
print("Classes:", test_dataset.class_names)
print("========================================")

Found 62 files belonging to 10 classes.

FINAL TEST DATASET
Test images: 64
Classes: ['Denver perfume', 'Dove HairFall Rescue', 'Livon serum', 'Lizol floor cleaner', 'Mamaearth Rosemary Water', 'clinic plus shampoo', 'nivya body mik', 'parachute hair oil', 'whitetone face powder', 'wotta girl perfume']


In [24]:
import os

valid_extensions = (".jpg", ".jpeg", ".png")

print("Non-JPG/PNG files remaining in test folder:\n")

for root, dirs, files in os.walk(test_dir):
    for f in files:
        if not f.lower().endswith(valid_extensions):
            print(os.path.join(root, f))

Non-JPG/PNG files remaining in test folder:



In [25]:
from PIL import Image
import os

bad_files = []
good_files = []

for root, dirs, files in os.walk(test_dir):
    for filename in files:

        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(root, filename)

            try:
                with Image.open(path) as img:
                    img.verify()

                good_files.append(path)

            except Exception as e:
                bad_files.append((path, str(e)))

print("Good images :", len(good_files))
print("Bad images  :", len(bad_files))

print("\nBAD FILES:")
for path, error in bad_files:
    print(path)

Good images : 62
Bad images  : 0

BAD FILES:


In [26]:
best_model = tf.keras.models.load_model(
    "RECYZA_BEST_10_EPOCH_MODEL.keras"
)

test_loss, test_accuracy = best_model.evaluate(
    test_dataset,
    verbose=1
)

print("\n========================================")
print("       RECYZA FINAL TEST RESULT")
print("========================================")
print(f"Test Images  : 62")
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print("========================================")

4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 795ms/step - accuracy: 0.9516 - loss: 0.3971

       RECYZA FINAL TEST RESULT
Test Images  : 62
Test Loss    : 0.3971
Test Accuracy: 95.16%


In [27]:

best_model.save("RECYZA_FINAL_MODEL_95_16.keras")

print("========================================")
print("RECYZA FINAL MODEL SAVED ✅")
print("========================================")
print("File: RECYZA_FINAL_MODEL_95_16.keras")
print("Test Accuracy: 95.16%")
print("Classes:", best_model.output_shape[-1])

RECYZA FINAL MODEL SAVED ✅
File: RECYZA_FINAL_MODEL_95_16.keras
Test Accuracy: 95.16%
Classes: 10
